### Simple preliminary_visit_image viewer with overlay
Creator: Andy Connolly

In [ ]:
from lsst.summit.utils.utils import checkStackSetup
checkStackSetup()

In [ ]:
import pylab as plt
import math
from lsst.daf.butler import Butler
import lsst.afw.display as afwDisplay
from lsst.ip.isr.isrTask import IsrTask
from lsst.cp.pipe.cpCombine import CalibCombineTask
from astropy.wcs import WCS
import traceback

In [ ]:
from astroquery.gaia import Gaia
import numpy as np
import lsst.geom


def find_gaia_sources(exposure):
    ''' Exposure with WCS query Gaia for sources'''
    wcs = exposure.info.getWcs()
    corners = exposure.getBBox().getCorners()
    
    corners2D = []
    for corner in corners:
        corners2D.append(lsst.geom.Point2D(corner))
    
    sky_corners = wcs.pixelToSky(corners2D)
    ra_sky_corners = np.array([point.getRa().asDegrees() for point in sky_corners])
    dec_sky_corners = np.array([point.getDec().asDegrees() for point in sky_corners])
    
    
    min_ra, max_ra = np.min(ra_sky_corners), np.max(ra_sky_corners)
    min_dec, max_dec = np.min(dec_sky_corners), np.max(dec_sky_corners)
    
    # Define a constraint to search for sources within the bounding box
    query = f"""
    SELECT ra, dec, phot_g_mean_mag  FROM gaiadr3.gaia_source
    WHERE ra BETWEEN {min_ra} AND {max_ra}
    AND dec BETWEEN {min_dec} AND {max_dec} ORDER BY phot_g_mean_mag
    """
    
    # Perform the query using astroquery
    job = Gaia.launch_job(query)
    return job.get_results()
    

In [ ]:
# on summit make this quickLook rather than nightlyValidation
butler = Butler('LSSTCam', collections=['LSSTCam/runs/quickLook'])

_day = 20250504
_seq_num = 355
_det = 92
dataRefs = butler.query_datasets('preliminary_visit_image' ,collections=['LSSTCam/runs/quickLook'],
                        where="instrument='LSSTCam' and exposure.day_obs={} and exposure.seq_num = {}\
                     and detector.id = {}".format(_day, _seq_num, _det))
exp = butler.get(dataRefs[0]) 

In [ ]:
# Activate interactive backend
%matplotlib widget

# Imports
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from mpl_toolkits.mplot3d import Axes3D
from ipywidgets import Dropdown, Button, VBox, Output, IntSlider, HBox, Checkbox
from IPython.display import display

# Create a consistent reference object to store our plot elements
class PlotState:
    def __init__(self):
        self.scatter = None

# Create an instance that persists across function calls
plot_state = PlotState()


def interactive_view(exp):
    # Sample image data (replace with your own data)
    image = exp.image.array
    wcs = exp.info.getWcs()
    wcs_header = wcs.getFitsMetadata()
    astropy_wcs = WCS(wcs_header)

    # Calculate the median value
    median_value = np.median(image)
    print(f"Median Value: {median_value}")

    # Close any previously open figures
    plt.close('all')
    # Parameters
    dx, dy = 100, 100  # Subarray size for surface plot. Make it 500 500 for donut images, 100 100 otherwise
    rmax = 100          # Max radial distance for radial plot
    grid_size = 10      # Grid size for printing values
    
    # Storage for clicked positions
    pos = []
    
    # Output widget for displaying results
    output = Output()
    
    # Set display stretch based on percentiles
    lower_percentile = 5
    upper_percentile = 70
    min_value = np.percentile(image, lower_percentile)
    max_value = np.percentile(image, upper_percentile)
    
    # Create figure and axis for the image display
    fig, ax = plt.subplots(subplot_kw={'projection': astropy_wcs}, figsize=(10, 10))

    im = ax.imshow(image, origin='lower', cmap='gray', vmin=min_value, vmax=max_value)

    # Overlay RA/Dec grid lines with finer labels
    ax.coords.grid(color='red', ls='dotted')
    #overlay gaia points
    gaia_check = Checkbox(
        value=False,       # Default state (unchecked)
        description='Overlay Gaia points',  # Label next to the checkbox
        disabled=False     # Allow the user to change the state
    )
    
    # Plot gaia sources
    def plot_gaia_sources(change):
        with output:
            if change['new']:  # When the checkbox is checked
                gaia = find_gaia_sources(exp)
                gaia=gaia[0:50]
                x_list = []
                y_list = []
                for ra, dec in zip(gaia["ra"], gaia["dec"]):
                    # Create a SpherePoint from RA and Dec
                    point = lsst.geom.SpherePoint(ra, dec, lsst.geom.degrees)
                    pixel = wcs.skyToPixel(point)
                    x_list.append(pixel.getX())
                    y_list.append(pixel.getY())
                plot_state.scatter = ax.scatter(
                    x_list, y_list, 
                    s=20, edgecolor='yellow', 
                    facecolor='none', label='Gaia Stars'
                )
            else:  # When the checkbox is unchecked
                if plot_state.scatter is not None:
                    plot_state.scatter.remove()
                    plot_state.scatter = None

            # Redraw the figure
            fig.canvas.draw_idle()
            #display(fig)


    # Register the callback function to be executed when the checkbox value changes
    gaia_check.observe(plot_gaia_sources, names='value')

    # Event handler
    def on_motion(event):
        # Check if the cursor is within the axes limits
        if event.inaxes == ax:
            # Get the x and y coordinates of the cursor
            x, y = int(event.xdata), int(event.ydata)
            
            # Ensure the coordinates are within the image bounds
            if 0 <= x < image.shape[1] and 0 <= y < image.shape[0]:
                # Get the pixel value at the current position
                pixel_value = image[y, x]
                wcs = exp.getWcs()
    
                ra, dec = wcs.pixelToSky(x,y)
                # Print the cursor position and pixel value
                with output:
                    output.clear_output()
                    print(f"Cursor at: x={x}, y={y}, ra={ra.asDegrees()}, dec={dec.asDegrees()}, Pixel Value={pixel_value:.2f}")
    
    fig.canvas.mpl_connect('motion_notify_event', on_motion)
    
    display(VBox([ HBox([ gaia_check]), output]))
    

In [ ]:
interactive_view(exp)
